# Oil Spill Detection - U-Net Training (Phase 2)

This notebook trains a 4-class U-Net model for detecting oil spills in SAR imagery.

**Classes:**
0. Sea (Background)
1. Oil Spill
2. Lookalike (Low-wind, biogenic film, etc.)
3. Ship

**Instructions for Google Colab:**
1. Ensure you have a GPU runtime enabled (`Runtime -> Change runtime type -> T4 GPU`).
2. Run the cells sequentially to download data, train the model, and save the weights.
3. Download the resulting `unet_spill_weights.pt` and place it in the `models/` directory of the project.

In [ ]:
!pip install torch torchvision rasterio opencv-python matplotlib shapely geopandas

## 1. Setup & Data Loading
In a real scenario, this cell downloads the Zenodo SAR Oil Spill Dataset.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import cv2
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create dummy dataset for demonstration
# (Replace this with actual Zenodo data loading in production)
class SyntheticSARDataset(Dataset):
    def __init__(self, num_samples=100, transform=None):
        self.num_samples = num_samples
        self.transform = transform

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random noise simulating SAR background using uint8 for OpenCV compatibility
        img = np.random.normal(loc=128, scale=25, size=(256, 256)).astype(np.uint8)
        mask = np.zeros((256, 256), dtype=np.uint8)
        
        # Add a synthetic oil spill (class 1)
        if np.random.rand() > 0.5:
            cx, cy = int(np.random.randint(50, 200)), int(np.random.randint(50, 200))
            rx, ry = int(np.random.randint(10, 50)), int(np.random.randint(10, 50))
            angle = int(np.random.randint(0, 180))
            cv2.ellipse(img, (cx, cy), (rx, ry), angle, 0, 360, (25,), -1)
            cv2.ellipse(mask, (cx, cy), (rx, ry), angle, 0, 360, (1,), -1)
            
        img_float = (img.astype(np.float32) / 255.0)
        img_tensor = torch.tensor(img_float).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long)
        
        return img_tensor, mask_tensor

train_dataset = SyntheticSARDataset(num_samples=500)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

## 2. Define the U-Net Model
This must match the architecture in `backend/detection/detector.py` exactly.

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4):
        super(UNet, self).__init__()
        
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True)
            )
            
        self.encoder1 = conv_block(in_channels, 64)
        self.encoder2 = conv_block(64, 128)
        self.encoder3 = conv_block(128, 256)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = conv_block(256, 512)
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = conv_block(512, 256)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = conv_block(256, 128)
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = conv_block(128, 64)
        
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))
        
        bottleneck = self.bottleneck(self.pool(enc3))
        
        dec3 = self.upconv3(bottleneck)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        
        return self.out_conv(dec1)

model = UNet(in_channels=1, out_channels=4).to(device)

## 3. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 5

print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for i, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

print("Training complete!")

## 4. Save Model Weights

In [ ]:
torch.save(model.state_dict(), 'unet_spill_weights.pt')
print("Model saved as 'unet_spill_weights.pt'. Please download this file and place it in your local 'models/' directory.")